# Predictive Maintenance AI - Exploratory Data Analysis & Modeling

**Project:** Predictive Maintenance AI  
**Dataset:** UCI AI4I 2020 Predictive Maintenance Dataset  
**Objective:** Predict industrial machine failure from sensor telemetry and operational variables for predictive maintenance.

## 1. Imports and Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

## 2. Load & Validate AI4I 2020 Dataset

In [ ]:
data_path = '../data/ai4i2020.csv'
if not os.path.exists(data_path):
    data_path = 'data/ai4i2020.csv'

df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape)
print('Missing Values:', df.isnull().sum().sum())
df.head()

## 3. Exploratory Data Analysis (EDA)

Analyze class imbalance and distribution across sensor features.

In [ ]:
# Machine failure distribution
counts = df['Machine failure'].value_counts()
print('Class Distribution:\n', counts)
print('\nPercentage:\n', df['Machine failure'].value_counts(normalize=True) * 100)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax[0], palette=['#10b981', '#ef4444'])
ax[0].set_title('Machine Failure Counts (0=Normal, 1=Failure)')
ax[0].set_xticklabels(['Normal (0)', 'Failure (1)'])

ax[1].pie(counts, labels=['Normal (96.6%)', 'Failure (3.4%)'], colors=['#10b981', '#ef4444'], autopct='%1.1f%%', startangle=140)
ax[1].set_title('Class Proportion (Imbalanced)')
plt.tight_layout()
plt.show()

In [ ]:
# Sensor feature distributions
sensor_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, col in enumerate(sensor_cols):
    sns.histplot(data=df, x=col, hue='Machine failure', kde=True, ax=axes[idx], palette=['#3b82f6', '#ef4444'], bins=30)
    axes[idx].set_title(f'Distribution of {col}')

axes[5].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
num_cols = sensor_cols + ['Machine failure']
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, square=True)
plt.title('Correlation Matrix of Sensor Variables and Machine Failure')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

Create physical domain features:
- `Temp_Difference`: Process temperature - Air temperature (heat dissipation)
- `Power_kW`: Torque * Rotational speed * (2 * pi / 60) / 1000 (mechanical power)
- `Overstrain_Index`: Tool wear * Torque (tool strain)
- `Torque_Speed_Ratio`: Torque / Rotational speed (stall / bind risk)

In [ ]:
def engineer_features(data):
    df_feat = data.copy()
    df_feat['Temp_Difference'] = df_feat['Process temperature [K]'] - df_feat['Air temperature [K]']
    df_feat['Power_kW'] = (df_feat['Torque [Nm]'] * (df_feat['Rotational speed [rpm]'] * 2 * np.pi / 60.0)) / 1000.0
    df_feat['Overstrain_Index'] = df_feat['Tool wear [min]'] * df_feat['Torque [Nm]']
    df_feat['Torque_Speed_Ratio'] = df_feat['Torque [Nm]'] / (df_feat['Rotational speed [rpm]'] + 1e-6)
    return df_feat

df_fe = engineer_features(df)
df_fe[['Temp_Difference', 'Power_kW', 'Overstrain_Index', 'Torque_Speed_Ratio']].describe()

## 5. Machine Learning Modeling & Comparison

Train Logistic Regression and Random Forest with class weighting and evaluate via Stratified 5-Fold Cross-Validation and Hold-out Test Set.

In [ ]:
all_features = ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temp_Difference', 'Power_kW', 'Overstrain_Index', 'Torque_Speed_Ratio']
num_features = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temp_Difference', 'Power_kW', 'Overstrain_Index', 'Torque_Speed_Ratio']
cat_features = ['Type']

X = df_fe[all_features]
y = df_fe['Machine failure']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_features)
])

models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    'Random Forest Classifier': Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=150, max_depth=10, min_samples_leaf=2, class_weight='balanced', random_state=42))
    ])
}

results = []
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_proba)
    
    results.append({
        'Model': name,
        'Accuracy': f'{acc*100:.2f}%',
        'Precision': f'{prec*100:.2f}%',
        'Recall': f'{rec*100:.2f}%',
        'F1-Score': f'{f1:.4f}',
        'ROC-AUC': f'{roc:.4f}'
    })

pd.DataFrame(results)

## 6. Confusion Matrix & Classification Report for Best Model

In [ ]:
rf_model = models['Random Forest Classifier']
y_pred_rf = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Pred Normal', 'Pred Failure'], yticklabels=['Actual Normal', 'Actual Failure'])
plt.title('Random Forest Confusion Matrix (Hold-out Test Set)')
plt.tight_layout()
plt.show()

print('Classification Report for Best Model (Random Forest):')
print(classification_report(y_test, y_pred_rf, target_names=['Normal (0)', 'Failure (1)'], digits=4))